# NB06: Blocked Reactions Analysis

Compare reaction flux ranges (FVA) between Marvin and OPAM2 deltaG values.
For each model, apply thermodynamic directionality constraints and run FVA to
identify reactions whose feasibility changes with the updated pKa values.

Reactions are classified as:
- **Blocked** (min=0, max=0) — no feasible flux
- **Forward** (min>=0, max>0)
- **Reverse** (min<0, max<=0)
- **Variable** (min<0, max>0) — reversible

In [1]:
import json
from pathlib import Path

import cobra
from cobra.flux_analysis import flux_variability_analysis
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

PROJECT_ROOT = Path('..').resolve()
DATA_DIR = PROJECT_ROOT / 'data'
FIGURES_DIR = PROJECT_ROOT / 'figures'

## 1. Load models and deltaG predictions

In [2]:
models = {}
for name in ['ecoli', 'bsubtilis']:
    models[name] = cobra.io.load_json_model(str(DATA_DIR / f'model_{name}.json'))
    m = models[name]
    sol = m.optimize()
    print(f'{name}: {len(m.reactions)} reactions, growth={sol.objective_value:.4f}')

with open(DATA_DIR / 'dg_predictions_marvin.json') as f:
    marvin_dg = json.load(f)
with open(DATA_DIR / 'dg_predictions_opam2.json') as f:
    opam2_dg = json.load(f)

print(f'\nMarvin dG predictions: {len(marvin_dg)}')
print(f'OPAM2 dG predictions: {len(opam2_dg)}')

ecoli: 1765 reactions, growth=0.5625


bsubtilis: 1345 reactions, growth=0.5000

Marvin dG predictions: 31924
OPAM2 dG predictions: 31924


## 2. Apply thermodynamic constraints and run FVA

Thermodynamic constraints: if deltaG > threshold, constrain to reverse only;
if deltaG < -threshold, constrain to forward only. We use |deltaG| > 5 kJ/mol
for confident directionality assignment.

In [3]:
DG_THRESHOLD = 5.0  # kJ/mol

def apply_thermo_constraints(model, dg_preds, threshold=DG_THRESHOLD):
    """Apply directional constraints from deltaG predictions."""
    constrained = 0
    for rxn in model.reactions:
        rxn_id = rxn.id.replace('_c0', '')
        if rxn_id in dg_preds:
            dg = dg_preds[rxn_id]['dG_mean']
            if dg > threshold:  # unfavorable forward
                rxn.upper_bound = min(rxn.upper_bound, 0)
                constrained += 1
            elif dg < -threshold:  # favorable forward
                rxn.lower_bound = max(rxn.lower_bound, 0)
                constrained += 1
    return constrained

def classify_fva(fva_df):
    """Classify reactions from cobra FVA DataFrame."""
    tol = 1e-9
    classes = {}
    for rxn_id in fva_df.index:
        mn = fva_df.loc[rxn_id, 'minimum']
        mx = fva_df.loc[rxn_id, 'maximum']
        if abs(mn) < tol and abs(mx) < tol:
            classes[rxn_id] = 'blocked'
        elif mn >= -tol and mx > tol:
            classes[rxn_id] = 'forward'
        elif mn < -tol and mx <= tol:
            classes[rxn_id] = 'reverse'
        else:
            classes[rxn_id] = 'variable'
    return classes

In [4]:
results = {}

for org_id, model in models.items():
    print(f'\n=== {org_id} ===')
    
    # Baseline FVA (no thermo constraints)
    fva_base = flux_variability_analysis(model, fraction_of_optimum=0.0)
    classes_base = classify_fva(fva_base)
    
    # Marvin-constrained FVA
    model_m = model.copy()
    n_m = apply_thermo_constraints(model_m, marvin_dg)
    fva_m = flux_variability_analysis(model_m, fraction_of_optimum=0.0)
    classes_m = classify_fva(fva_m)
    
    # OPAM2-constrained FVA
    model_o = model.copy()
    n_o = apply_thermo_constraints(model_o, opam2_dg)
    fva_o = flux_variability_analysis(model_o, fraction_of_optimum=0.0)
    classes_o = classify_fva(fva_o)
    
    print(f'  Thermo constraints: Marvin={n_m}, OPAM2={n_o}')
    for label, cls in [('Baseline', classes_base), ('Marvin', classes_m), ('OPAM2', classes_o)]:
        counts = pd.Series(cls).value_counts()
        print(f'  {label}: {dict(counts)}')
    
    results[org_id] = {
        'classes_base': classes_base,
        'classes_marvin': classes_m,
        'classes_opam2': classes_o,
        'n_constrained_marvin': n_m,
        'n_constrained_opam2': n_o,
    }


=== ecoli ===


  Thermo constraints: Marvin=928, OPAM2=926
  Baseline: {'blocked': np.int64(1139), 'forward': np.int64(363), 'variable': np.int64(158), 'reverse': np.int64(105)}
  Marvin: {'blocked': np.int64(1732), 'variable': np.int64(17), 'forward': np.int64(9), 'reverse': np.int64(7)}
  OPAM2: {'blocked': np.int64(1731), 'variable': np.int64(20), 'forward': np.int64(8), 'reverse': np.int64(6)}

=== bsubtilis ===


  Thermo constraints: Marvin=731, OPAM2=734
  Baseline: {'blocked': np.int64(1007), 'forward': np.int64(178), 'variable': np.int64(88), 'reverse': np.int64(72)}
  Marvin: {'blocked': np.int64(1337), 'variable': np.int64(5), 'reverse': np.int64(2), 'forward': np.int64(1)}
  OPAM2: {'blocked': np.int64(1337), 'variable': np.int64(5), 'reverse': np.int64(2), 'forward': np.int64(1)}


## 3. Compare blocked reactions: Marvin vs OPAM2

In [5]:
diff_records = []
summary_records = []

for org_id, res in results.items():
    cm = res['classes_marvin']
    co = res['classes_opam2']
    all_rxns = set(cm.keys()) & set(co.keys())
    
    newly_blocked = 0
    newly_unblocked = 0
    direction_changed = 0
    
    for rxn_id in all_rxns:
        c_m, c_o = cm[rxn_id], co[rxn_id]
        if c_m == c_o:
            continue
        
        rxn_seed = rxn_id.replace('_c0', '')
        diff_records.append({
            'organism': org_id,
            'reaction': rxn_id,
            'marvin_class': c_m,
            'opam2_class': c_o,
            'marvin_dg': marvin_dg.get(rxn_seed, {}).get('dG_mean'),
            'opam2_dg': opam2_dg.get(rxn_seed, {}).get('dG_mean'),
        })
        
        if c_m != 'blocked' and c_o == 'blocked':
            newly_blocked += 1
        elif c_m == 'blocked' and c_o != 'blocked':
            newly_unblocked += 1
        else:
            direction_changed += 1
    
    summary_records.append({
        'organism': org_id,
        'total_rxns': len(all_rxns),
        'blocked_marvin': sum(1 for v in cm.values() if v == 'blocked'),
        'blocked_opam2': sum(1 for v in co.values() if v == 'blocked'),
        'newly_blocked': newly_blocked,
        'newly_unblocked': newly_unblocked,
        'direction_changed': direction_changed,
    })
    
    print(f'{org_id}: +{newly_blocked} blocked, -{newly_unblocked} unblocked, {direction_changed} direction changes')

df_diff = pd.DataFrame(diff_records)
df_summary = pd.DataFrame(summary_records)
print(f'\nTotal classification changes: {len(df_diff)}')
print(df_summary.to_string(index=False))

ecoli: +2 blocked, -3 unblocked, 0 direction changes
bsubtilis: +0 blocked, -0 unblocked, 0 direction changes

Total classification changes: 5
 organism  total_rxns  blocked_marvin  blocked_opam2  newly_blocked  newly_unblocked  direction_changed
    ecoli        1765            1732           1731              2                3                  0
bsubtilis        1345            1337           1337              0                0                  0


## 4. Growth impact under thermodynamic constraints

In [6]:
growth_results = []

for org_id, model in models.items():
    sol_base = model.optimize()
    
    model_m = model.copy()
    apply_thermo_constraints(model_m, marvin_dg)
    sol_m = model_m.optimize()
    
    model_o = model.copy()
    apply_thermo_constraints(model_o, opam2_dg)
    sol_o = model_o.optimize()
    
    growth_results.append({
        'organism': org_id,
        'growth_unconstrained': round(sol_base.objective_value, 6),
        'growth_marvin': round(sol_m.objective_value, 6),
        'growth_opam2': round(sol_o.objective_value, 6),
    })
    print(f'{org_id}: unconstrained={sol_base.objective_value:.4f}, marvin={sol_m.objective_value:.4f}, opam2={sol_o.objective_value:.4f}')

df_growth = pd.DataFrame(growth_results)
print(df_growth.to_string(index=False))

ecoli: unconstrained=0.5625, marvin=-0.0000, opam2=-0.0000


bsubtilis: unconstrained=0.5000, marvin=0.0000, opam2=0.0000
 organism  growth_unconstrained  growth_marvin  growth_opam2
    ecoli                0.5625           -0.0          -0.0
bsubtilis                0.5000            0.0           0.0


## 5. Visualize

In [7]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for idx, (org_id, res) in enumerate(results.items()):
    ax = axes[idx]
    categories = ['blocked', 'forward', 'reverse', 'variable']
    marvin_counts = [sum(1 for v in res['classes_marvin'].values() if v == c) for c in categories]
    opam2_counts = [sum(1 for v in res['classes_opam2'].values() if v == c) for c in categories]
    
    x = np.arange(len(categories))
    width = 0.35
    ax.bar(x - width/2, marvin_counts, width, label='Marvin', color='#2196F3')
    ax.bar(x + width/2, opam2_counts, width, label='OPAM2', color='#FF9800')
    ax.set_xlabel('Reaction Classification')
    ax.set_ylabel('Count')
    ax.set_title(org_id)
    ax.set_xticks(x)
    ax.set_xticklabels(categories)
    ax.legend()
    
    for bars in ax.containers:
        ax.bar_label(bars, fontsize=8, padding=2)

plt.suptitle('Reaction Classification: Marvin vs OPAM2 Thermodynamic Constraints')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'blocked_reactions_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved figure to figures/blocked_reactions_comparison.png')

Saved figure to figures/blocked_reactions_comparison.png


## 6. Save results

In [8]:
if len(df_diff) > 0:
    df_diff.to_csv(DATA_DIR / 'blocked_reactions_diff.tsv', sep='\t', index=False)
    print(f'Saved {len(df_diff)} reaction changes to data/blocked_reactions_diff.tsv')

df_growth.to_csv(DATA_DIR / 'growth_comparison.tsv', sep='\t', index=False)
df_summary.to_csv(DATA_DIR / 'fva_summary.tsv', sep='\t', index=False)
print('Saved growth and FVA summary')

Saved 5 reaction changes to data/blocked_reactions_diff.tsv
Saved growth and FVA summary
